In [2]:
# =========================
# INIT — Fine-tune WavLM + LoRA (STRICT) + auto-detect paths + HF cache = ROOT/hf_cache
# =========================

import os, random, warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ---------- helpers ----------
def pick_existing(paths: List[Path]) -> Path:
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("Tidak ada path yang ditemukan:\n" + "\n".join(map(str, paths)))

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    import torch
    import torch.nn as nn
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception as e:
    raise RuntimeError("PyTorch belum siap. Pastikan torch terinstall dan bisa diimport.") from e

# ---------- auto-detect ROOT (ikut pola notebook baseline_strict kamu) ----------
CWD = Path.cwd().resolve()
ROOT = None
for p in [CWD] + list(CWD.parents):
    if (p / "output" / "split_strict" / "manifest_strict.csv").exists():
        ROOT = p
        break
    if (p / "split_strict" / "manifest_strict.csv").exists():  # fallback
        ROOT = p
        break

if ROOT is None:
    raise FileNotFoundError(
        "Gagal nemu ROOT. Pastikan ada 'output/split_strict/manifest_strict.csv' "
        "di salah satu parent folder dari notebook kamu."
    )

# ---------- key dirs (ikut struktur hasil notebook kamu) ----------
STRICT_DIR = pick_existing([
    ROOT / "output" / "split_strict",
    ROOT / "split_strict",
])
MANIFEST_STRICT = STRICT_DIR / "manifest_strict.csv"

PREP_DIR = pick_existing([
    ROOT / "output" / "preprocessing",
    ROOT / "preprocessing",
])
FULL_WAV_DIR = pick_existing([
    PREP_DIR / "preprocessed_full",
])

VAD_DROP = PREP_DIR / "vad" / "vad_drop.csv"
if not VAD_DROP.exists():
    raise FileNotFoundError(f"vad_drop.csv tidak ditemukan di: {VAD_DROP}")

# ---------- HF cache (ROOT/hf_cache) ----------
HF_CACHE_DIR = ROOT / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE_DIR / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# import transformers + peft setelah env cache diset
try:
    from transformers import AutoFeatureExtractor, WavLMModel
    from peft import LoraConfig, TaskType, get_peft_model
except Exception as e:
    raise RuntimeError(
        "Library belum lengkap. Pastikan 'transformers' dan 'peft' terinstall."
    ) from e

# ---------- load strict manifest + VAD drop filter ----------
df_all = pd.read_csv(MANIFEST_STRICT)
df_all["clip_id"] = df_all["clip_id"].astype(str)

if "split_strict" not in df_all.columns:
    if "split" in df_all.columns:
        df_all = df_all.rename(columns={"split": "split_strict"})
    else:
        raise ValueError(f"Kolom split_strict tidak ditemukan. Kolom tersedia: {df_all.columns.tolist()}")

drop_df = pd.read_csv(VAD_DROP)
drop_ids = set(drop_df["clip_id"].astype(str).tolist())

df_all = df_all[~df_all["clip_id"].isin(drop_ids)].reset_index(drop=True)

spl = df_all["split_strict"].astype(str).str.lower()
df_train = df_all[spl == "train"].reset_index(drop=True)
df_val   = df_all[spl == "val"].reset_index(drop=True)
df_test  = df_all[spl == "test"].reset_index(drop=True)  # kunci test, jangan dipakai tuning

# audio path untuk training (pakai output/preprocessing/preprocessed_full/<clip_id>.wav)
df_train["audio_path"] = df_train["clip_id"].map(lambda x: str(FULL_WAV_DIR / f"{x}.wav"))
df_val["audio_path"]   = df_val["clip_id"].map(lambda x: str(FULL_WAV_DIR / f"{x}.wav"))
df_test["audio_path"]  = df_test["clip_id"].map(lambda x: str(FULL_WAV_DIR / f"{x}.wav"))

# quick check audio exists (sampling)
for name, df_ in [("train", df_train), ("val", df_val), ("test", df_test)]:
    sample = df_.sample(min(5, len(df_)), random_state=SEED) if len(df_) else df_
    ok = [Path(p).exists() for p in sample["audio_path"].tolist()] if len(sample) else []
    if len(ok) and not all(ok):
        raise FileNotFoundError(f"Ada audio_path sample yang tidak ada di {name}. Cek FULL_WAV_DIR: {FULL_WAV_DIR}")
    
# hard check semua audio ada (kalau kamu mau strict banget)
missing_train = df_train[~df_train["audio_path"].map(lambda p: Path(p).exists())]
missing_val   = df_val[~df_val["audio_path"].map(lambda p: Path(p).exists())]
missing_test  = df_test[~df_test["audio_path"].map(lambda p: Path(p).exists())]
assert len(missing_train) == 0 and len(missing_val) == 0 and len(missing_test) == 0, \
    f"Masih ada audio yang missing. train={len(missing_train)} val={len(missing_val)} test={len(missing_test)}"


# ---------- labels ----------
label_cols = ["extraversion", "neuroticism", "agreeableness", "conscientiousness", "openness"]
NUM_LABELS = len(label_cols)

# pastikan label float
for c in label_cols:
    df_train[c] = df_train[c].astype("float32")
    df_val[c]   = df_val[c].astype("float32")
    df_test[c]  = df_test[c].astype("float32")

# ---------- model config (match baseline frozen kamu) ----------
MODEL_NAME = "microsoft/wavlm-base-plus"

# LoRA (fase 1: rank tetap dulu)
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

LR = 1e-4
WEIGHT_DECAY = 0.01

# ---------- build model (freeze backbone + attach LoRA + regression head) ----------
feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_NAME,
    cache_dir=str(HF_CACHE_DIR),
    # kalau mau wajib pakai cache (tanpa download), aktifkan ini:
    # local_files_only=True
)

backbone = WavLMModel.from_pretrained(
    MODEL_NAME,
    cache_dir=str(HF_CACHE_DIR),
    # local_files_only=True
)


for p in backbone.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)
backbone = get_peft_model(backbone, lora_cfg)

class WavLMRegressor(nn.Module):
    def __init__(self, backbone: WavLMModel, num_labels: int = 5):
        super().__init__()
        self.backbone = backbone
        h = backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(h, h),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(h, num_labels),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, input_values, attention_mask=None, labels=None):
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        x = out.last_hidden_state  # (B,T,H)

        if attention_mask is None:
            pooled = x.mean(dim=1)
        else:
            m = attention_mask.unsqueeze(-1).to(x.dtype)
            pooled = (x * m).sum(dim=1) / (m.sum(dim=1).clamp(min=1.0))

        preds = self.head(pooled)
        loss = self.loss_fn(preds, labels) if labels is not None else None
        return {"loss": loss, "preds": preds}

model = WavLMRegressor(backbone, NUM_LABELS).to(DEVICE)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

# ---------- output dirs ----------
RUN_NAME = f"wavlm_strict_lora_r{LORA_R}_lr{LR}_seed{SEED}"
OUT_ROOT = ROOT / "output" / "finetune_strict" / RUN_NAME
CKPT_DIR = OUT_ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

tot = sum(p.numel() for p in model.parameters())
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Init OK")
print("ROOT         :", ROOT)
print("HF_CACHE_DIR :", HF_CACHE_DIR)
print("MANIFEST     :", MANIFEST_STRICT)
print("VAD_DROP     :", VAD_DROP)
print("FULL_WAV_DIR :", FULL_WAV_DIR)
print("Split shapes :", "train", df_train.shape, "| val", df_val.shape, "| test", df_test.shape, "(locked)")
print("MODEL_NAME   :", MODEL_NAME)
print("label_cols   :", label_cols)
print("DEVICE       :", DEVICE)
print(f"params total={tot:,} | trainable={trn:,}")
print("OUT_ROOT     :", OUT_ROOT)
print("CKPT_DIR     :", CKPT_DIR)


Init OK
ROOT         : E:\tugas-akhir-qiqi
HF_CACHE_DIR : E:\tugas-akhir-qiqi\hf_cache
MANIFEST     : E:\tugas-akhir-qiqi\output\split_strict\manifest_strict.csv
VAD_DROP     : E:\tugas-akhir-qiqi\output\preprocessing\vad\vad_drop.csv
FULL_WAV_DIR : E:\tugas-akhir-qiqi\output\preprocessing\preprocessed_full
Split shapes : train (5936, 15) | val (1999, 15) | test (2039, 15) (locked)
MODEL_NAME   : microsoft/wavlm-base-plus
label_cols   : ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
DEVICE       : cpu
params total=95,271,285 | trainable=889,349
OUT_ROOT     : E:\tugas-akhir-qiqi\output\finetune_strict\wavlm_strict_lora_r8_lr0.0001_seed42
CKPT_DIR     : E:\tugas-akhir-qiqi\output\finetune_strict\wavlm_strict_lora_r8_lr0.0001_seed42\checkpoints


In [3]:
# =========================
# DATASET + DATALOADER (WavLM strict finetune)
# =========================

import torch
from torch.utils.data import Dataset, DataLoader

import numpy as np

# audio IO
import soundfile as sf

# optional resample
try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except Exception:
    _HAS_TORCHAUDIO = False

TARGET_SR = getattr(feature_extractor, "sampling_rate", 16000)
print("Target SR:", TARGET_SR, "| torchaudio:", _HAS_TORCHAUDIO)

class StrictAudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_cols, target_sr=16000):
        self.df = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.target_sr = target_sr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = row["audio_path"]

        audio, sr = sf.read(wav_path)  # audio: (T,) or (T,C)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)  # to mono

        audio = audio.astype(np.float32)

        # resample if needed
        if sr != self.target_sr:
            if not _HAS_TORCHAUDIO:
                raise RuntimeError(
                    f"Sample rate beda (got {sr}, expected {self.target_sr}) tapi torchaudio tidak tersedia."
                )
            x = torch.from_numpy(audio).unsqueeze(0)  # (1,T)
            x = torchaudio.functional.resample(x, sr, self.target_sr)
            audio = x.squeeze(0).numpy().astype(np.float32)

        labels = row[self.label_cols].to_numpy(dtype=np.float32)  # (5,)
        return {"audio": audio, "labels": labels}

def collate_fn(batch):
    audios = [b["audio"] for b in batch]
    labels = torch.tensor([b["labels"] for b in batch], dtype=torch.float32)

    # padding + attention_mask
    feats = feature_extractor(
        audios,
        sampling_rate=TARGET_SR,
        padding=True,
        return_tensors="pt",
    )
    feats["labels"] = labels
    return feats

BATCH_SIZE = 4  # CPU: kecilin dulu biar aman
NUM_WORKERS = 0 # Windows aman 0 dulu

train_ds = StrictAudioDataset(df_train, label_cols, target_sr=TARGET_SR)
val_ds   = StrictAudioDataset(df_val, label_cols, target_sr=TARGET_SR)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=(DEVICE=="cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=(DEVICE=="cuda")
)

# quick sanity check 1 batch
batch = next(iter(train_loader))
print({k: tuple(v.shape) for k,v in batch.items() if hasattr(v, "shape")})


Target SR: 16000 | torchaudio: True
{'input_values': (4, 240000), 'attention_mask': (4, 240000), 'labels': (4, 5)}
